# Grad-CAM — Best BraTS 2D Baseline (End-to-End ResNet50)

Loads the best `brats_png` baseline from `leaderboard.json`  
(`e50_b8_resnet50_cat_lr5e-05_cv5_decay0.001`, F1=0.404, 3-class categorical).

Displays a **2×4 figure** for four patients:
- **Top row** — T2 scan (q50 slice) + GT segmentation mask overlay (BraTS colour scheme)
- **Bottom row** — T2 scan + Grad-CAM heatmap overlay

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from datasets.brats_png_os import BraTSPNGDatasetWithOS

print('Imports OK')

In [ ]:
# ── Configuration — edit here ────────────────────────────────────────────────
RUN_DIR = Path(
    '/path/to/BrainWear_Kareem/FYP'
    '/baseline/end-to-end/runs/brats_png'
    '/e50_b8_resnet50_cat_lr5e-05_cv5_decay0.001'
)
FOLD            = 1      # 1–5; fold 1 has highest accuracy (0.468)
TARGET_CLASS    = None   # None → predicted class; int → force specific class
DISPLAY_CHANNEL = 2      # T2 quantile channel (0=q25 … 4=q75; 2=q50 median)

# Override with explicit val-set indices to pick specific patients, e.g. [0, 3, 7, 12]
# None → auto-pick 4 correct predictions (round-robin across classes)
PATIENT_IDXS = None

# ── Display options ───────────────────────────────────────────────────────────
SHOW_LEGEND    = False   # show GT segmentation colour key (top row)
SHOW_COLORBAR  = False   # show Grad-CAM activation scale (bottom row)

# BraTS segmentation colour scheme (matches visualisation_2d.ipynb)
BRATS_CMAP = mcolors.ListedColormap(['black', 'red', 'yellow', 'blue'])
BRATS_BOUNDS = [-0.5, 0.5, 1.5, 2.5, 3.5]
BRATS_NORM   = mcolors.BoundaryNorm(BRATS_BOUNDS, BRATS_CMAP.N)
GT_ALPHA     = 0.45

# Quantile suffix for the seg mask file (must match DISPLAY_CHANNEL)
_QUANTILE_SUFFIXES = ['q25', 'q38', 'q50', 'q62', 'q75']
SEG_SUFFIX = _QUANTILE_SUFFIXES[DISPLAY_CHANNEL]   # 'q50'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Run:    {RUN_DIR.name}')
print(f'Seg suffix: {SEG_SUFFIX}')
print(f'Show legend: {SHOW_LEGEND}  |  Show colorbar: {SHOW_COLORBAR}')

In [ ]:
# ── Load checkpoint and rebuild model ───────────────────────────────────────
with open(RUN_DIR / 'args.json') as f:
    args = json.load(f)
print('Run args:', json.dumps(args, indent=2))

ckpt_path = RUN_DIR / f'best_model_fold{FOLD}.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'
ckpt = torch.load(ckpt_path, map_location=device)
num_outputs = ckpt['num_outputs']
print(f'\nnum_outputs: {num_outputs}  |  best_epoch: {ckpt["best_epoch"]}  |  val_loss: {ckpt["best_val_loss"]:.4f}')

if args['model'] == 'resnet18':
    model = tvm.resnet18(weights=None)
else:
    model = tvm.resnet50(weights=None)

in_ch = args['in_channels']
model.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc    = nn.Linear(model.fc.in_features, num_outputs)
model.load_state_dict(ckpt['model_state_dict'])
model.eval().to(device)
print(f'Model ({args["model"]}) loaded.')

In [ ]:
# ── Load dataset and reconstruct the exact validation split ─────────────────
dataset = BraTSPNGDatasetWithOS(
    root_dir=args['data_dir'],
    score_file=args['csv_path'],
    num_bins=args['num_bins'],
    quantile_bins=args['quantile'],
    shuffle=False,
    max_patients=args.get('max_patients'),
)
print(f'Dataset: {len(dataset)} patients')

all_idx    = np.arange(len(dataset))
all_labels = np.array(dataset.get_class_labels())

fold_seed = args.get('fold_seed') or args['seed']
n_splits  = args['n_splits']
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=fold_seed)
fold_splits = list(skf.split(all_idx, all_labels))

_, val_idx_np = fold_splits[FOLD - 1]
val_indices   = val_idx_np.tolist()
print(f'Fold {FOLD}: {len(val_indices)} validation patients')
print(f'True labels (val): {[int(all_labels[i]) for i in val_indices]}')

In [ ]:
# ── Grad-CAM ─────────────────────────────────────────────────────────────────
def compute_gradcam(model, input_tensor, target_class=None):
    """
    Grad-CAM for a single 2D multi-channel input.

    Args:
        model        : eval-mode torchvision ResNet on device
        input_tensor : (1, C, H, W) float tensor on device
        target_class : class index; None → argmax prediction

    Returns:
        cam        : (H, W) numpy array in [0, 1]
        pred_class : int
    """
    activations, gradients = [], []
    target_layer = model.layer4[-1]

    fwd = target_layer.register_forward_hook(
        lambda m, i, o: activations.append(o)
    )
    bwd = target_layer.register_full_backward_hook(
        lambda m, gi, go: gradients.append(go[0])
    )

    try:
        model.zero_grad()
        logits = model(input_tensor)                              # (1, num_classes)
        pred_class = int(logits.argmax(dim=1).item())

        tc = target_class if target_class is not None else pred_class
        one_hot = torch.zeros_like(logits)
        one_hot[0, tc] = 1.0
        logits.backward(gradient=one_hot)

        act  = activations[0]                                     # (1, C, h, w)
        grad = gradients[0]                                       # (1, C, h, w)
        weights = grad.mean(dim=(2, 3), keepdim=True)             # (1, C, 1, 1)
        cam = (weights * act).sum(dim=1, keepdim=True)            # (1, 1, h, w)
        cam = F.relu(cam)

        H, W = input_tensor.shape[2:]
        cam = F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()                # (H, W)

        vmin, vmax = float(cam.min()), float(cam.max())
        cam = (cam - vmin) / (vmax - vmin + 1e-8)
    finally:
        fwd.remove()
        bwd.remove()

    return cam, pred_class


def load_seg_mask(data_dir, patient_id, suffix):
    """Load a BraTS PNG segmentation mask as integer array (values 0–3)."""
    path = Path(data_dir) / patient_id / f'{patient_id}_seg_{suffix}.png'
    return np.array(Image.open(path), dtype=np.int32)


print('compute_gradcam() and load_seg_mask() defined.')

In [ ]:
# ── Select 4 correct predictions (round-robin across classes) ────────────────
N_PATIENTS = 4

if PATIENT_IDXS is not None:
    selected = list(PATIENT_IDXS)
    assert len(selected) == N_PATIENTS, f'PATIENT_IDXS must contain exactly {N_PATIENTS} entries'
else:
    # Pre-run inference over all val patients to identify correct predictions
    correct_class_to_val_idx = defaultdict(list)
    with torch.no_grad():
        for vi in val_indices:
            tensor, label = dataset[vi]
            inp = tensor.unsqueeze(0).to(device)
            pred = int(model(inp).argmax(dim=1).item())
            if pred == int(all_labels[vi]):
                correct_class_to_val_idx[int(all_labels[vi])].append(vi)

    print('Correct predictions per class:')
    for c in sorted(correct_class_to_val_idx.keys()):
        print(f'  class {c}: {len(correct_class_to_val_idx[c])} patients')

    selected = []
    classes  = sorted(correct_class_to_val_idx.keys())
    ci = 0
    while len(selected) < N_PATIENTS:
        c = classes[ci % len(classes)]
        if correct_class_to_val_idx[c]:
            selected.append(correct_class_to_val_idx[c].pop(0))
        ci += 1

print('\nSelected dataset indices:', selected)
for s in selected:
    print(f'  [{s}]  {dataset.patient_folders[s]}  class={int(all_labels[s])}')

In [ ]:
# ── Run Grad-CAM and load GT masks ───────────────────────────────────────────
results = []
for idx in selected:
    patient_id = dataset.patient_folders[idx]
    tensor, label = dataset[idx]                          # (5, H, W), int
    inp = tensor.unsqueeze(0).to(device)                  # (1, 5, H, W)

    cam, pred = compute_gradcam(model, inp, TARGET_CLASS)
    img = tensor[DISPLAY_CHANNEL].numpy()                 # (H, W) — q50 T2 slice
    seg = load_seg_mask(args['data_dir'], patient_id, SEG_SUFFIX)  # (H, W) int 0–3

    results.append({
        'idx':        idx,
        'patient_id': patient_id,
        'img':        img,
        'seg':        seg,
        'cam':        cam,
        'true':       int(label),
        'pred':       pred,
    })
    tick = '\u2713' if int(label) == pred else '\u2717'
    has_tumour = np.any(seg > 0)
    print(f'  [{idx}] {patient_id}  true={int(label)}  pred={pred}  {tick}  tumour_in_slice={has_tumour}')

In [ ]:
# ── 2×4 figure: top row = GT overlay, bottom row = Grad-CAM overlay ─────────
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for col, r in enumerate(results):
    img_norm = r['img'].astype(np.float32)
    mx = img_norm.max()
    if mx > 0:
        img_norm /= mx

    tick  = '✓' if r['true'] == r['pred'] else '✗'
    # col_title = f"{r['patient_id']}\nTrue: {r['true']}   Pred: {r['pred']}   {tick}"
    col_title = f""

    # ── Top row: scan + GT segmentation mask ─────────────────────────────────
    axes[0, col].imshow(img_norm, cmap='gray')
    gt_overlay = np.ma.masked_where(r['seg'] == 0, r['seg'])
    axes[0, col].imshow(gt_overlay, cmap=BRATS_CMAP, norm=BRATS_NORM,
                        alpha=GT_ALPHA, interpolation='none')
    axes[0, col].set_title(col_title, fontsize=8)
    axes[0, col].axis('off')

    # ── Bottom row: scan + Grad-CAM heatmap ──────────────────────────────────
    axes[1, col].imshow(img_norm, cmap='gray')
    gcam_im = axes[1, col].imshow(r['cam'], cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[1, col].axis('off')

# Row labels
axes[0, 0].set_ylabel('GT mask', fontsize=10, labelpad=6)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=10, labelpad=6)
for ax in axes[:, 0]:
    ax.yaxis.set_visible(True)
    ax.tick_params(left=False, labelleft=False)

# Grad-CAM colorbar
if SHOW_COLORBAR:
    cbar_gcam = fig.colorbar(gcam_im, ax=axes[1, :].tolist(),
                              orientation='vertical', fraction=0.015, pad=0.02)
    cbar_gcam.set_label('Grad-CAM activation', fontsize=9)

# GT legend
if SHOW_LEGEND:
    legend_patches = [
        mpatches.Patch(color='red',    label='1 — Necrotic core'),
        mpatches.Patch(color='yellow', label='2 — Peritumoral edema'),
        mpatches.Patch(color='blue',   label='3 — Enhancing tumour'),
    ]
    axes[0, -1].legend(
        handles=legend_patches, loc='upper left',
        bbox_to_anchor=(1.02, 1), fontsize=8, framealpha=0.9,
    )

fig.suptitle(
    f'BraTS Baseline (top: GT mask | bottom: GradCAM)',
    fontsize=12, fontweight='bold', y=1.01,
)
plt.tight_layout()

SAVE_PATH = FYP_ROOT / 'eval' / f'gradcam_brats_2d_new.png'
plt.savefig(SAVE_PATH, dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_PATH}')
plt.show()

In [ ]:
# ── Slot Attention model + combined 4×(2+k) figure ───────────────────────────
from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D

# ── Config — point this at any SlotClassifier2D checkpoint ───────────────────
SA_EXP_NAME = "v14a_0.15_test"
SA_CKPT_PATH = "/path/to/BrainWear_Kareem/FYP/slot_attention/training_2d/models/checkpoints/brats_png_v14a_0.15_test/ckpt.pt"

sa_ckpt = torch.load(SA_CKPT_PATH, map_location=device)
hp = sa_ckpt['hyperparameters']
sa_model = SlotClassifier2D(
    in_shape=(hp['in_channels'], hp['input_h'], hp['input_w']),
    width=hp['width'],
    num_slots=hp['num_slots'],
    slot_dim=hp['slot_dim'],
    routing_iters=hp['routing_iters'],
    temperature=hp['temp'],
    encoder_depth=hp.get('encoder_depth', 4),
    enc3_init_skip=hp.get('enc3_init_skip', False),
    use_mask_pool_classifier=hp.get('use_mask_pool_classifier', False),
)
if 'model_state_dict' in sa_ckpt:
    sa_model.load_state_dict(sa_ckpt['model_state_dict'])
else:
    sa_model.load_state_dict(sa_ckpt)
sa_model.to(device).eval()
sa_model.set_deterministic_slot_init(seed=42)
num_slots = hp['num_slots']
print(f"Slot Attention model loaded: {SA_EXP_NAME}  ({num_slots} slots)")

# ── Run slot attention on the q50 T2 slice for each selected patient ─────────
for r in results:
    tensor, _ = dataset[r['idx']]
    # Slot model expects single-channel input: (1, 1, H, W)
    t2_slice = tensor[DISPLAY_CHANNEL].unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, masks, _, _ = sa_model(t2_slice)
    r['slot_masks'] = masks[0, :, 0].cpu().numpy()   # (num_slots, H, W)

# ── 4×(2+k) figure: GT | Grad-CAM | slot_1 … slot_k ─────────────────────────
cols = 2 + num_slots
rows = len(results)
fig2, axes2 = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

col_titles = ['GT mask', 'Grad-CAM'] + [f'Slot {i+1}' for i in range(num_slots)]
for c, title in enumerate(col_titles):
    axes2[0, c].set_title(title, fontsize=10)

for row, r in enumerate(results):
    img_norm = r['img'].astype(np.float32)
    mx = img_norm.max()
    if mx > 0:
        img_norm /= mx

    # Column 0: scan + GT segmentation overlay
    axes2[row, 0].imshow(img_norm, cmap='gray')
    gt_overlay = np.ma.masked_where(r['seg'] == 0, r['seg'])
    axes2[row, 0].imshow(gt_overlay, cmap=BRATS_CMAP, norm=BRATS_NORM,
                         alpha=GT_ALPHA, interpolation='none')
    axes2[row, 0].axis('off')

    # Column 1: scan + Grad-CAM overlay
    axes2[row, 1].imshow(img_norm, cmap='gray')
    axes2[row, 1].imshow(r['cam'], cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes2[row, 1].axis('off')

    # Columns 2+: slot attention masks (viridis, same scale as visualisation_2d.ipynb)
    for i in range(num_slots):
        axes2[row, i + 2].imshow(r['slot_masks'][i], cmap='viridis', vmin=0, vmax=1)
        axes2[row, i + 2].axis('off')

plt.tight_layout()
SAVE_PATH2 = FYP_ROOT / 'eval' / f'gradcam_slots_brats_2d_{SA_EXP_NAME}.png'
plt.savefig(SAVE_PATH2, dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_PATH2}')
plt.show()

In [ ]:
# ── 4×(1+k) figure: GT | slot_1 … slot_k (no Grad-CAM) ──────────────────────
cols = 1 + num_slots
rows = len(results)
fig3, axes3 = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

col_titles = ['GT mask'] + [f'Slot {i+1}' for i in range(num_slots)]
for c, title in enumerate(col_titles):
    axes3[0, c].set_title(title, fontsize=10)

for row, r in enumerate(results):
    img_norm = r['img'].astype(np.float32)
    mx = img_norm.max()
    if mx > 0:
        img_norm /= mx

    # Column 0: scan + GT segmentation overlay
    axes3[row, 0].imshow(img_norm, cmap='gray')
    gt_overlay = np.ma.masked_where(r['seg'] == 0, r['seg'])
    axes3[row, 0].imshow(gt_overlay, cmap=BRATS_CMAP, norm=BRATS_NORM,
                         alpha=GT_ALPHA, interpolation='none')
    axes3[row, 0].axis('off')

    # Columns 1+: slot attention masks
    for i in range(num_slots):
        axes3[row, i + 1].imshow(r['slot_masks'][i], cmap='viridis', vmin=0, vmax=1)
        axes3[row, i + 1].axis('off')

plt.tight_layout()
SAVE_PATH3 = FYP_ROOT / 'eval' / f'gt_slots_brats_2d_{SA_EXP_NAME}.png'
plt.savefig(SAVE_PATH3, dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_PATH3}')
plt.show()